In [4]:
import sys
import gzip
from datetime import datetime
from alignment.PoseGraph import PoseGraph

import google.protobuf
import delimited_protobuf
from protocol import message_formats_pb2, telemetry_pb2, control_pb2, req_rep_pb2
import numpy as np

In [2]:
with (gzip.open("../logs/19_04_2024/BYEDP210004_ee6c4d5b1b0969d4_00514.bez", "rb") as bez, 
      gzip.open("../logs/19_04_2024/multibeam_BYEDP210004_2024-04-19_132753.995.bez", "rb") as mbez):
    while True:
        try:
            binlog = delimited_protobuf.read(bez, message_formats_pb2.BinlogRecord)
            if telemetry_pb2.PositionEstimateTel.DESCRIPTOR.full_name not in binlog.payload.type_url:
                continue
            telemetry = telemetry_pb2.PositionEstimateTel()
            telemetry.ParseFromString(binlog.payload.value)
            
            binlog.clock_monotonic.seconds
     
            print(telemetry)
            # ping = telemetry_pb2.OculusPingTel()
            # ping.ParseFromString(binlog.payload.value)
            # print(ping.ping, binlog.clock_monotonic.seconds + binlog.clock_monotonic.nanos / 1e9)
        except Exception as e:
            print(e)
            break  

position_estimate {
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
  }
}

position_estimate {
  northing: -0.09047472476959229
  easting: 0.7818724513053894
  heading: 0.018644850701093674
  surge_rate: 0.0001990995806409046
  sway_rate: 0.00010730788198998198
  yaw_rate: 2.148166367010873e-11
  ocean_current: 2.8394308628776344e-06
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
    is_valid: true
  }
}

position_estimate {
  northing: 8.292933671327773e-06
  easting: 0.00031964894151315093
  heading: 0.0007127672433853149
  surge_rate: -0.0006337274680845439
  sway_rate: 0.00024311395827680826
  yaw_rate: -4.430056581988806e-12
  ocean_current: 0.0008807017002254725
  global_position {
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
    is_valid: true
  }
}

position_estimate {
  northing: -0.00022728634939994663
  easting: 0.00042534683598

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [5]:
def read_binlog(file_path, type_names):
    with gzip.open(file_path, "rb") as fin:
        while True:
            try:
                binlog = delimited_protobuf.read(fin, message_formats_pb2.BinlogRecord)
                found = None
                for type_name in type_names:
                    if type_name.DESCRIPTOR.full_name in binlog.payload.type_url:
                        found = type_name
                
                if not found:
                    continue
                
                message = found()
                message.ParseFromString(binlog.payload.value)
                yield message, binlog.clock_monotonic.seconds + binlog.clock_monotonic.nanos / 1e9, binlog.unix_timestamp.seconds + binlog.unix_timestamp.nanos / 1e9
            except Exception as ex:
                print("Terminating loop:", ex)
                return


In [13]:
import g2o
from alignment.plot_slam2d import plot_slam2d

generator = read_binlog("../logs/19_04_2024/BYEDP210004_ee6c4d5b1b0969d4_00514.bez", [req_rep_pb2.SyncTimeReq, telemetry_pb2.PositionEstimateTel])

time_offset = 0
time_offset_monotonic = 0

pose_graph = PoseGraph(verbose=True)

pose_graph.add_fixed_pose(g2o.SE2())

count = 0
last_time = 0
freq = 0
for message, monotonic, unix in generator:
    # Correct the timestamps
    monotonic = unix
    unix += time_offset
    
    if isinstance(message, req_rep_pb2.SyncTimeReq):
        time_offset = message.time.unix_timestamp.seconds
        time_offset_monotonic = monotonic
        print("Time offset:", time_offset)
        
        print(datetime.fromtimestamp(time_offset))
        continue
        
    if time_offset == 0:
        continue
        
    if isinstance(message, telemetry_pb2.PositionEstimateTel):
        if not message.position_estimate.is_valid:
            continue
            
    if count % 20 == 0:
        freq = 20 / (unix - last_time)
        last_time = unix
        print(f"Frequency: {freq} Hz")
        print(message)
    # pose_graph.add_odometry(
    #         message.position_estimate.northing,
    #         message.position_estimate.easting,
    #         message.position_estimate.heading,
    #         np.eye(3))
    count += 1
    if count > 2000:
        break
    


# fig = plot_slam2d(pose_graph.optimizer, "Before optimisation")
# fig.write_image("before_optimisation.png")
# fig.write_html("before_optimisation.html")
# fig.show("notebook")
    


    > PoseGraph: Adding fixed pose vertex with ID 0
Time offset: 1713529807
2024-04-19 14:30:07
Time offset: 1713529826
2024-04-19 14:30:26
Time offset: 1713530694
2024-04-19 14:44:54
Time offset: 1713530792
2024-04-19 14:46:32
Frequency: 1.1671787400834626e-08 Hz
position_estimate {
  northing: 0.1405280977487564
  easting: -0.11085373908281326
  heading: 0.3452320992946625
  surge_rate: 0.021920161321759224
  sway_rate: 0.004512503277510405
  yaw_rate: 0.00026724525378085673
  ocean_current: 0.24407771229743958
  odometer: 0.732746958732605
  is_valid: true
  global_position {
    latitude: 63.44100126069645
    longitude: 10.417997778789452
  }
  navigation_sensors {
    sensor_id: NAVIGATION_SENSOR_ID_WATERLINKED_DVL_A50
    is_valid: true
  }
}

Frequency: 1.9872810909600378 Hz
position_estimate {
  northing: 0.3374382257461548
  easting: 0.010551669634878635
  heading: 0.525900661945343
  surge_rate: -0.0011436311760917306
  sway_rate: 0.001072532031685114
  yaw_rate: -0.00041053

In [11]:
generator = read_binlog("../logs/19_04_2024/multibeam_BYEDP210004_2024-04-19_132753.995.bez", [telemetry_pb2.OculusPingTel])

time_offset = 0
time_offset_monotonic = 0

pose_graph = PoseGraph(verbose=True)

pose_graph.add_fixed_pose(g2o.SE2())

count = 0
for message, monotonic, unix in generator:
    print(message)
    break
    

    > PoseGraph: Adding fixed pose vertex with ID 0
ping {
  ping_id: 22088
  ping_firing_date: 4810
  range: 20.406503677368164
  gain: 49.661242961883545
  frequency: 1196808.5106382978
  speed_of_sound_used: 1481.8412895039141
  range_resolution: 0.0396738307909848
  temperature: 7.0
  pressure: 0.28076171875
  master_mode: 1
  has_gains: true
  number_of_ranges: 514
  number_of_beams: 256
  step: 260
  sample_size: 1
  bearings: -6500
  bearings: -6405
  bearings: -6313
  bearings: -6224
  bearings: -6138
  bearings: -6054
  bearings: -5972
  bearings: -5893
  bearings: -5815
  bearings: -5738
  bearings: -5663
  bearings: -5590
  bearings: -5518
  bearings: -5447
  bearings: -5378
  bearings: -5309
  bearings: -5242
  bearings: -5176
  bearings: -5111
  bearings: -5046
  bearings: -4983
  bearings: -4920
  bearings: -4858
  bearings: -4797
  bearings: -4736
  bearings: -4676
  bearings: -4617
  bearings: -4559
  bearings: -4501
  bearings: -4444
  bearings: -4387
  bearings: -4331